In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]

In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

In [ ]:
fig, axs, error_stats = plot_empirical_model_error_distribution(
    flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    plot_mode="hist2d",
    split_by_subgroup=True,
    show_split_marginals=True,
    bins=50,
    title="Empirical Error vs Observed Ventilation by Opening Type",
    cmap="magma_r",
    error_xlabel=r"Error; $\sigma_{\Delta C_p}$ Bulk Fit PN",
    ventilation_ylabel="LES Flux Velocity",
    positive_ylabel="Flow Entering",
    negative_ylabel="Flow Exiting",
    print_counts=True,
    colorbar_max=40,
)
display(
    pd.DataFrame(
        [
            {
                "opening_type": opening_type,
                "mu_hat": np.nan if stats is None else stats["mu_hat"],
                "sigma_hat": np.nan if stats is None else stats["sigma_hat"],
                "n": 0 if stats is None else stats["n"],
            }
            for opening_type, stats in error_stats.items()
        ]
    )
)

fig, axs, error_stats = plot_empirical_model_error_distribution(
    flowStatsMI,
    y_var=y_var,
    x_var="q_model-Norm-Adjusted",
    plot_mode="hist2d",
    split_by_subgroup=True,
    show_split_marginals=True,
    bins=50,
    title="Empirical Error vs Observed Ventilation by Opening Type",
    cmap="magma_r",
    error_xlabel=r"Error; $\sigma_{\Delta C_p}$ Bulk Fit PN",
    ventilation_ylabel="LES Flux Velocity",
    positive_ylabel="Flow Entering",
    negative_ylabel="Flow Exiting",
    print_counts=True,
    colorbar_max=40
)


In [ ]:
from pathlib import Path

from emulationHelpers import (
    fit_bayesian_ventilation_q_subgroups,
    fit_bayesian_ventilation_p_subgroups,
    load_bayesian_q_ventilation_fit_results,
    load_bayesian_ventilation_fit_results,
    plot_bayesian_ventilation_q_fit_results,
    plot_bayesian_ventilation_p_fit_results,
    plot_bayesian_ventilation_parameter_posteriors,
    plot_bayesian_ventilation_parameter_qq,
    plot_bayesian_ventilation_parameter_traces,
    save_bayesian_ventilation_fit_results,
    summarize_bayesian_parameter_normal_fit_tests,
    summarize_bayesian_ventilation_fits,
)

a_mu = 1.0
a_sigma = 0.1
p_rms_var = "p_rms-noInt-Norm"
obs_sigma = 0.01

sample_kwargs = {
    "draws": 1000,
    "tune": 1000,
    "chains": 4,
    "cores": 4,
    "progressbar": True,
}

cache_dir = Path("mcmc_cache") / "pressure_scalar_posteriors_log_p_rms"
rerun_mcmc = True

if cache_dir.exists() and not rerun_mcmc:
    bayes_fits = load_bayesian_ventilation_fit_results(cache_dir)
    print(f"Loaded cached Bayesian fits from {cache_dir}")
else:
    bayes_fits = fit_bayesian_ventilation_p_subgroups(
        data=flowStatsMI,
        y_var=y_var,
        x_var=x_var,
        p_rms_var=p_rms_var,
        a_mu=a_mu,
        a_sigma=a_sigma,
        obs_sigma=obs_sigma,
        sample_kwargs=sample_kwargs,
        random_seed=42,
    )
    save_bayesian_ventilation_fit_results(bayes_fits, cache_dir)
    print(f"Saved Bayesian fits to {cache_dir}")


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian Pressure Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
)

bayes_summary = summarize_bayesian_ventilation_fits(bayes_fits)
display(bayes_summary)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.45,
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_posteriors(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
    kde=False,
    overlay_normal=True,
    normal_line_color="tab:blue",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_traces(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
plt.show()


In [ ]:
bayes_normal_fit_shapiro = summarize_bayesian_parameter_normal_fit_tests(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
display(bayes_normal_fit_shapiro)


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_qq(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
plt.show()


In [ ]:
# Bayesian q_rms workflow for $\\sigma_{q_n}$ Bulk Measured PN
q_rms_var = "rms-mass_flux-Norm"

q_cache_dir = Path("mcmc_cache") / "q_rms_scalar_posteriors_log_q_rms"
rerun_q_mcmc = True

if q_cache_dir.exists() and not rerun_q_mcmc:
    q_bayes_fits = load_bayesian_q_ventilation_fit_results(q_cache_dir)
    print(f"Loaded cached Bayesian q fits from {q_cache_dir}")
else:
    q_bayes_fits = fit_bayesian_ventilation_q_subgroups(
        data=flowStatsMI,
        y_var=y_var,
        x_var=x_var,
        q_rms_var=q_rms_var,
        a_mu=a_mu,
        a_sigma=a_sigma,
        obs_sigma=obs_sigma,
        sample_kwargs=sample_kwargs,
        random_seed=42,
    )
    save_bayesian_ventilation_fit_results(q_bayes_fits, q_cache_dir)
    print(f"Saved Bayesian q fits to {q_cache_dir}")


In [ ]:
fig, axs = plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian q_rms Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
)

q_bayes_summary = summarize_bayesian_ventilation_fits(q_bayes_fits)
display(q_bayes_summary)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.45,
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_posteriors(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
    kde=False,
    overlay_normal=True,
    normal_line_color="tab:blue",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_traces(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
plt.show()


In [ ]:
q_bayes_normal_fit_shapiro = summarize_bayesian_parameter_normal_fit_tests(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
display(q_bayes_normal_fit_shapiro)


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_qq(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
plt.show()


## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/windowASHRAE.csv")
data=flowStatsMI

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")